# langnostic colab runner (t4)

model work runs here: dataset download, whisper transcription, lora training.
only `results/cache/*.jsonl` (+ `.meta.json`) comes back to the laptop.

runtime → change runtime type → **t4 gpu**.


In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
import os
REPO = 'https://github.com/Coding-With-Josh/langnostic.git'
if not os.path.isdir('/content/langnostic'):
    !git clone $REPO /content/langnostic
%cd /content/langnostic
!git pull --ff-only

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
# one naija shard covers training; the rest are test-only
!PYTHONPATH=. python -m speechgap.data build --out-dir data \
    --naija yoruba:0-0 \
    --voxpopuli en:test \
    --l2arctic \
    --fleurs yo_ng:test --allow-clip-level
!PYTHONPATH=. python -m speechgap.data verify --out-dir data

In [ ]:
import json
audit = json.load(open('data/audit.json'))
print(json.dumps({k: audit[k] for k in ('splits', 'groups', 'dropped', 'accents_seen', 'problems')}, indent=2))

In [ ]:
# zero-shot baseline
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --out results/cache/zero_shot.jsonl

# rerun with --resume after a disconnect

In [ ]:
!PYTHONPATH=. python -m speechgap.report results/cache/zero_shot.jsonl \
    --out results/zero_shot_report.json

## lora: pilot first

`speechgap.train` publishes an adapter only if its final validation passes the
gates: wer, wrong script, degenerate output, forced language. it prints three
ref/hyp pairs to check. a failing pilot means the 15/30-minute runs are not worth it.


In [ ]:
# check=True stops on a failed gate
import os, subprocess, sys
os.environ['PYTHONPATH'] = '.'

def run(*args):
    print('$', ' '.join(args), flush=True)
    subprocess.run([sys.executable, '-m', *args], check=True)

run('speechgap.train', '--train', 'data/manifest_train.jsonl',
    '--val', 'data/manifest_val.jsonl', '--minutes', '2', '--epochs', '1',
    '--max-val', '20', '--patience', '1', '--out', 'checkpoints/yo-pilot',
    '--group', 'yoruba_read', '--language', 'yo')

In [ ]:
# set after reading the pilot output
PILOT_REVIEWED = False

if not PILOT_REVIEWED:
    raise SystemExit('read the pilot output first, then set PILOT_REVIEWED = True')

for minutes in (15, 30):
    run('speechgap.train', '--train', 'data/manifest_train.jsonl',
        '--val', 'data/manifest_val.jsonl', '--minutes', str(minutes),
        '--group', 'yoruba_read', '--language', 'yo',
        '--out', f'checkpoints/yo-{minutes}min')

In [ ]:
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --adapter checkpoints/yo-15min --out results/cache/yo_lora15.jsonl
!PYTHONPATH=. python -m speechgap.eval --manifest data/manifest_test.jsonl \
    --device cuda --adapter checkpoints/yo-30min --out results/cache/yo_lora30.jsonl

for name in ('zero_shot', 'yo_lora15', 'yo_lora30'):
    !PYTHONPATH=. python -m speechgap.report results/cache/$name.jsonl --out results/${name}_report.json

In [ ]:
# transcripts only: no audio, manifests or weights
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/langnostic/cache
!cp results/cache/*.jsonl results/cache/*.meta.json /content/drive/MyDrive/langnostic/cache/

## next

`speechgap.train` reads `data/manifest_train.jsonl` and `data/manifest_val.jsonl`,
trains speaker-disjoint from test, and writes adapters to `checkpoints/` (not
committed). then score every adapter on every group.
